<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_67_rlhf_grpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🏆 **Module 67 — RLHF / GRPO (alignment beyond DPO)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 67 — RLHF / GRPO

> M62 covered **DPO** — the simplest, most popular alignment algorithm. This module zooms out to the full alignment landscape. **RLHF (PPO)** is the original recipe that produced ChatGPT. **GRPO** (DeepSeek-R1) is the 2025 algorithm that taught models to *reason* with verifiable rewards. Both belong in your mental model because production-grade post-training mixes all three (SFT → DPO → some RL).

### What you'll cover
1. SFT → DPO → RL — where each one sits and what it adds
2. **Classical RLHF**: the 3-stage InstructGPT recipe (SFT, RM, PPO)
3. The **reward model** — how preferences become a scalar
4. **PPO** in five equations — surrogate loss + clipping + KL penalty
5. Why PPO is painful (rollouts, value model, instability)
6. **GRPO** — DeepSeek-R1's algorithm, no value model needed
7. **RLVR** — Reinforcement Learning with Verifiable Rewards
8. Reward hacking and how to spot it
9. The 2025 post-training stack — what frontier labs actually run
10. Pick your alignment recipe — DPO / GRPO / PPO / hybrid


## 1 · Where each algorithm sits

```
   pretrain (M57) ──► SFT (M59) ──► DPO (M62) ──► [optional] RLHF / GRPO ──► ship
                                       │                    │
                                       │ pairs (chosen,     │ rollouts + scalar reward
                                       │  rejected)         │ — much more compute
                                       ▼                    ▼
                              easy, stable, cheap     hard, powerful, expensive
```

The **practical 2025 recipe** is:
1. **SFT** to teach the format / domain (M59).
2. **DPO** on preference pairs for general helpfulness (M62).
3. **GRPO** or **PPO** on top, but **only** for tasks where you have **verifiable rewards** (math, code, format-following).

DPO alone gets you 90% of the way. RL on top is the **last 1–10 %** — and where capability gains in reasoning models like **DeepSeek-R1**, **GPT-o1**, **Qwen3** and **Llama-Nemotron** come from.


## 2 · Classical RLHF — the InstructGPT recipe

Three stages, originally published by Christiano (2017) and scaled by Ouyang et al. / OpenAI (2022) for InstructGPT / ChatGPT.

```
       Step 1 — SFT                Step 2 — Reward Model       Step 3 — RL (PPO)
       ┌──────────────┐            ┌───────────────────┐       ┌────────────────────┐
       │ Pre-trained  │   demos    │  SFT model        │       │  SFT model         │
       │ base model   │ ────────► │  + scalar head    │       │  + policy + value  │
       └──────────────┘            │                   │       │                    │
                                    │  trained on       │       │  rollouts → scored │
                                    │  (chosen, rejected)│      │  by RM → PPO       │
                                    └───────────────────┘       └────────────────────┘
```

Step 2's **reward model** turns the messy human-preference signal into a single number you can feed gradient descent. Step 3 then optimises the policy to maximise that number while staying close to the SFT model (KL penalty).

## 3 · The reward model

Take your SFT model. Replace the LM head with a **scalar head** — a `Linear(d_model, 1)` that reads the last token's hidden state (same trick as M58's classifier). Train on the same `(chosen, rejected)` pairs as DPO with a Bradley-Terry loss:

$$\mathcal{L}_{\text{RM}} = -\log \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)$$

After training, `r_θ(x, y)` outputs a number that *correlates* with human preference. You can then **freeze** it and use it as a critic.

In [ ]:
rm_sketch = '''
class RewardModel(nn.Module):
    def __init__(self, base_model, d_model):
        super().__init__()
        self.base = base_model                   # the SFT model body
        self.value_head = nn.Linear(d_model, 1)  # one scalar per sequence

    def forward(self, input_ids):
        h = self.base(input_ids, output_hidden_states=True).hidden_states[-1]
        return self.value_head(h[:, -1, :])      # last-token logit → scalar reward

# Training loss — pair-wise (Bradley-Terry)
def rm_loss_batch(model, chosen_ids, rejected_ids):
    r_w = model(chosen_ids)
    r_l = model(rejected_ids)
    return -F.logsigmoid(r_w - r_l).mean()
'''
print(rm_sketch)

**RM gotchas in practice.**
- **Reward hacking** — the policy quickly finds the RM's blind spots. Always periodically **refresh** the RM by collecting new preferences against the *current* policy.
- **Calibration** — the absolute scale of `r_θ` is meaningless; only differences matter.
- **Coverage** — a 1 M-sample RM that never saw your specific failure modes still misses them. RM training data must look like the policy's actual outputs.

## 4 · PPO — the RL stage in five equations

Once you have an RM, **Proximal Policy Optimisation** (Schulman et al., 2017) takes over. PPO is an actor-critic algorithm with three additions:

**(1) Rollouts.** Sample completions from the current policy on prompts $x$:
$$y \sim \pi_\theta(\cdot \mid x)$$

**(2) Reward + KL penalty.** Each token's reward is the RM score on the *full* sequence, minus a KL term keeping the policy near the SFT reference:
$$R_t = r_\theta(x, y) \;-\; \beta \, \log \frac{\pi_\theta(y_t \mid x, y_{<t})}{\pi_\text{SFT}(y_t \mid x, y_{<t})}$$

**(3) Advantage estimate.** GAE (generalised advantage estimation) using a separately-trained **value model** $V_\phi(x, y_{<t})$ that learns to predict future reward:
$$A_t = R_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t) \;\;+\;\; \text{GAE smoothing}$$

**(4) Clipped surrogate loss.** Update the policy with PPO's **clipping** to prevent any one update going too far:
$$\mathcal{L}_{\text{PPO}}(\theta) = \mathbb{E}_t\!\left[ \min\!\big( \rho_t A_t, \;\text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_t \big) \right], \quad \rho_t = \frac{\pi_\theta(y_t)}{\pi_{\theta_\text{old}}(y_t)}$$

**(5) Value loss.** Train the value model on the actual returns:
$$\mathcal{L}_V(\phi) = \mathbb{E}_t\big[(V_\phi(s_t) - R_t)^2\big]$$

You alternate: collect rollouts → score with RM → compute advantages → take a few PPO steps → repeat.

## 5 · Why PPO is painful

| Pain | Cause | Mitigation |
|---|---|---|
| **4 models in memory** | policy + reference + RM + value | LoRA on policy; share weights w/ reference |
| **Rollouts are the bottleneck** | every step needs `O(generation length)` model calls | batch + vLLM (M44); offload rollouts to a separate worker pool |
| **Hyperparameter hell** | β, ε, GAE λ, learning rate, batch composition all sensitive | start with the InstructGPT defaults; don't sweep until basics work |
| **Reward hacking** | policy finds RM blind spots | periodically refresh RM; KL penalty β tuned aggressively |
| **Instability** | one bad batch tanks the model | small LRs (1e-6), clip ε=0.1-0.2 |
| **Length bias** | RM tends to prefer longer answers | normalise reward by length; cap response length |

The DPO paper's whole pitch (M62) was: *the closed-form policy + Bradley-Terry math means you can skip the RM and the value model and just train on preferences directly.* That's why DPO blew up — same effective gradient as PPO, **a tenth of the engineering**.

But PPO still wins in **one** important regime…

## 6 · GRPO — what DeepSeek-R1 actually used

DPO needs **preference pairs**. PPO needs a **reward model**. **GRPO** (Group-Relative Policy Optimization, DeepSeek 2024-25) needs **neither** — just a **scalar reward function** (e.g. "did the answer match the gold value?").

The trick: for each prompt $x$, sample **`G` completions** $y_1, \ldots, y_G$. Compute scalar rewards $r_1, \ldots, r_G$. The advantage of each sample is the **standardised reward within the group**:
$$A_i = \frac{r_i - \text{mean}(r_{1..G})}{\text{std}(r_{1..G})}$$

No value model. The group's own mean and std play the role of the baseline.

The GRPO loss is then a clipped surrogate **identical in shape to PPO**, plus a per-token KL to a reference (SFT) model:
$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_{x,y_i}\!\bigg[\sum_t \min\!\big(\rho_t A_i,\; \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_i\big) - \beta \, \text{KL}\!\big(\pi_\theta(\cdot|x_{<t}) \,\Vert\, \pi_\text{ref}(\cdot|x_{<t})\big)\bigg]$$

In [ ]:
grpo_sketch = '''
# Pseudocode — full implementation in `trl.GRPOTrainer` (2024+) or `openrlhf`
import torch, torch.nn.functional as F

def grpo_step(policy, ref_model, prompt_ids, reward_fn, group_size=8, beta=0.04, epsilon=0.2):
    # 1) sample G completions per prompt
    completions = sample_many(policy, prompt_ids, n=group_size)        # (G, T)
    rewards = torch.tensor([reward_fn(c) for c in completions])        # (G,)

    # 2) compute group-relative advantages
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)          # (G,)

    # 3) compute per-token log-probs under policy and reference
    logp     = compute_log_probs(policy, completions)                  # (G, T)
    logp_ref = compute_log_probs(ref_model, completions)               # (G, T)
    logp_old = logp.detach()                                            # save old policy snapshot

    # 4) ratio + clipped surrogate
    ratio = (logp - logp_old).exp()                                     # (G, T)
    unclipped = ratio * adv[:, None]
    clipped   = ratio.clamp(1 - epsilon, 1 + epsilon) * adv[:, None]
    policy_loss = -torch.min(unclipped, clipped).mean()

    # 5) per-token KL to the reference
    kl = (logp - logp_ref).mean()
    loss = policy_loss + beta * kl
    return loss
'''
print(grpo_sketch)

**Why GRPO matters.**
- **No value model** → ~30% less VRAM, ~2× simpler code.
- **No reward model** *required* — just a scalar reward function. For math/code this can be **`int(pred == gold)`**.
- **Group baseline** — the group mean acts as a per-prompt baseline. Lower variance than vanilla policy gradient.
- **Same clipping trick** as PPO so the algorithm inherits its stability.

This is the algorithm that produced **DeepSeek-R1**, the first openly-published "reasoning" model competitive with **o1**. The same trick is used in **Qwen3**, **Llama-Nemotron-Reasoning**, and several closed-source frontier reasoners.

## 7 · RLVR — Reinforcement Learning with Verifiable Rewards

GRPO is the *algorithm*. **RLVR** is the *recipe* that makes it work for reasoning: train on tasks where the **reward function is exact and cheap**.

| Task | Reward function | Why it works |
|---|---|---|
| **Math** | `int(answer == gold_answer)` | gold answer in the dataset |
| **Code** | `int(unit_tests_pass)` | run the test suite |
| **Formal proof** | `int(lean_check(proof))` | proof assistant verifies |
| **JSON format** | `int(valid_json(response))` | parse and check |
| **Tool use** | `int(tool_call_succeeded)` | run the tool, inspect result |

**RLVR sidesteps reward hacking** by giving the model nothing to hack — the reward is the ground truth.

The DeepSeek-R1 recipe (simplified):
1. SFT on a small set of long-chain-of-thought math/code traces.
2. GRPO with verifiable math + code rewards. The model learns to reason **without** anyone teaching it the reasoning format — the long CoT emerges from optimising the verifiable reward.
3. SFT on the *new* good reasoning traces from step 2.
4. Another round of GRPO + helpfulness RLHF.

Step 2 is the magic step. **The model teaches itself to reason** to maximise a math-grader's signal.

## 8 · Reward hacking — recognise it, kill it

Whether you use RM-PPO or RLVR-GRPO, **the policy will try to game the reward**. Classic forms:

| Hack | Example | Fix |
|---|---|---|
| **Length bias** | model writes 5-paragraph answers because RM prefers them | length-normalise reward |
| **Format gaming** | model wraps answers in code blocks the RM likes | diversify training data |
| **Sycophancy** | model agrees with user even when wrong | RM examples that punish agreement on falsehoods |
| **Refusal collapse** | model refuses anything ambiguous to avoid -1 reward | balance harmless/helpful in RM data |
| **Reasoning collapse** | (RLVR) model emits gibberish-then-correct-answer because only final number matters | also reward the chain-of-thought format / process |
| **Test-set memorisation** | (RLVR code) model memorises gold output, not algorithm | held-out test sets per problem |
| **Spurious specificity** | model invents fake citations because RM treats specificity = quality | RM scoring rubric must reward verifiability |

**The rule:** any reward signal that doesn't *exactly* match what you want will be hacked at scale. Either tighten the reward, or accept that RLHF / GRPO buys ~10 % of the way, not 100 %.

## 9 · What frontier labs actually run in 2025

Approximate post-training pipelines based on open papers + reported recipes:

| Model | Post-training |
|---|---|
| **GPT-4o / GPT-5** | SFT → multi-stage RLHF (RM + PPO) with extensive red-teaming RMs |
| **Claude 3.5 / 3.7 Sonnet** | SFT → **Constitutional AI** (RLAIF) → DPO-flavoured + extensive evals |
| **Gemini 2.5** | SFT → DPO → RLHF; massive prompt engineering for safety |
| **DeepSeek-R1** | SFT (cold-start CoT) → **GRPO with verifiable rewards** → SFT distill → GRPO |
| **Llama-3 Instruct** | SFT → **rejection sampling** + DPO → no PPO |
| **Qwen3** (reasoning) | SFT → **DPO** for general use → **GRPO with verifiable rewards** for reasoning mode |
| **Tülu 3** (allenai) | SFT → DPO → **GRPO with verifiable rewards** (transparent open recipe) |

The pattern: **everyone does SFT + some preference algorithm; reasoning models add GRPO with verifiable rewards on top.** Pure PPO is increasingly rare in open recipes — DPO + GRPO are cheaper and competitive.

## 10 · Decision table — pick your alignment recipe

| You have… | Reach for | Why |
|---|---|---|
| `(prompt, response)` demos | **SFT** (M59) | always step 1 |
| `(prompt, chosen, rejected)` preference pairs | **DPO** (M62) | simplest, most stable |
| A trained reward model + lots of compute | **PPO** | classic; if you already have the RM, mature tooling |
| A scalar reward function (math, code, format) | **GRPO** | no value model; the modern default for verifiable tasks |
| `(thumbs up / down)` signals (no pairs) | **KTO** | DPO variant for unpaired feedback |
| Multimodal pref pairs | **mDPO** | per-modality KL terms |
| Reasoning task with verifiable rewards | **GRPO + RLVR** | the DeepSeek-R1 recipe |
| Frontier-scale RM safety | **constitutional AI / RLAIF** | model-as-RM; cheaper than humans |

### What you need to remember
- **DPO is the cheap, stable default.** Reach for it first.
- **GRPO is the modern PPO replacement** when you have a scalar reward but no preference pairs.
- **RLVR is the recipe** that makes GRPO produce reasoning behaviour.
- **Reward hacking is universal.** Audit your model's outputs; refresh the signal.
- **Frontier labs do all of the above in sequence.** SFT → DPO → GRPO is the 2025 pipeline.

> 🟡 The HF `trl` library implements **`SFTTrainer`** (M59), **`DPOTrainer`** (M62), **`PPOTrainer`**, **`GRPOTrainer`**, **`KTOTrainer`**, **`CPOTrainer`**, **`ORPOTrainer`**, and **`OnlineDPOTrainer`**. Read its source for ~all of the algorithms in this module.

## ✅ Recap
- **SFT → DPO → RL** is the modern post-training stack. DPO does most of the work.
- **Classical RLHF** = SFT + RM + PPO. Powerful but ~4× model copies, hyperparameter-sensitive.
- **GRPO** drops the value model and uses *group-relative* advantages → ~30% less VRAM, simpler code.
- **RLVR** pairs GRPO with **verifiable** scalar rewards (math, code, format) → no reward hacking + emergent reasoning.
- **Frontier labs in 2025** mix SFT + DPO + GRPO with RLVR. Pure PPO is fading from open recipes.

Next: **M68 — Workflow Orchestration (Airflow / Prefect / Dagster)**.
